In [1]:
import os
import gymnasium as gym
import numpy as np
import pybullet as p
import pybullet_data
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from noise import pnoise2

from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv, VecNormalize
from stable_baselines3.common.logger import configure
from stable_baselines3.common.callbacks import CheckpointCallback

In [2]:
URDF_PATH = r"D:\Project Arbeit\Steps-Generation-Using-Reinforcement-Learning\URDF Model\a1.urdf"
SAVE_PATH = r"D:\Project Arbeit\Steps-Generation-Using-Reinforcement-Learning\Project Arbeit Model"
LOGS_PATH = r"D:\Project Arbeit\Steps-Generation-Using-Reinforcement-Learning\Project Arbeit Logs"
TRAINING_MODEL = r"D:\Project Arbeit\Steps-Generation-Using-Reinforcement-Learning\Project Arbeit Model\Training Model"



if not os.path.exists(SAVE_PATH):
    os.makedirs(SAVE_PATH)

if not os.path.exists(TRAINING_MODEL):
    os.makedirs(TRAINING_MODEL)

# Constants to define training and visualisation.
GUI_MODE = True      # Set "True" to display pybullet in a window
EPISODE_LENGTH = 1000      # Number of steps for one training episode
MAXIMUM_LENGTH = 20_000_000     # Number of total steps for entire training
SIZE_OBSERVATION = 42
BOUND_ANGLE_MIN = np.array([-0.802851455917, -1.0471975512, -2.69653369433, -0.802851455917, -1.0471975512, -2.69653369433, -0.802851455917, -1.0471975512, -2.69653369433, -0.802851455917, -1.0471975512, -2.69653369433]) # Joint minimum angle (rad)
BOUND_ANGLE_MAX = np.array([0.802851455917, 4.18879020479, -0.916297857297, 0.802851455917, 4.18879020479, -0.916297857297, 0.802851455917, 4.18879020479, -0.916297857297, 0.802851455917, 4.18879020479, -0.916297857297]) # Joint maximum angle (rad)


In [3]:
class OpenCatGymEnv(gym.Env): ## We have created a custom class which inherits from the Env class of gym libraray
    """ Gymnasium environment (stable baselines 3) for OpenCat robots.
    """

    metadata = {'render.modes': ['human']} ## Just to show what kind of visualization we want. (there are 2-3 other modes also like rgb_array)

    def __init__(self, target_speed = 0.375): ## Constructor to setup everything
        self.episode_length = 1000
        self.control_frequency = 100  # Hz
        self.target_speed = target_speed
        self.step_counter = 0  # Counts steps in the current episode
        self.step_counter_session = 0 # May be used to track steps across all episodes (total training time).
        self.alpha1 = 0.04
        self.alpha2 = 20
        self.alive_bonus = 20 * target_speed
        self.vx_smooth = 0
        self.vy_smooth = 0
        self.wyaw_smooth = 0
        self.alpha_smooth = 0.2
        self.prev_action = np.zeros(12)
        self.max_vel = 25
        self.prev_torque = np.zeros(12)
        # In __init__:
        self.contact_avg = np.array([0.5, 0.5, 0.5, 0.5])
        self.work_avg = np.zeros(4)

        
        
        self.default_pose = np.array([
                    -0.05, 0.8, -1.6,
                    0.05, 0.8, -1.6,
                    -0.05, 0.8, -1.6,
                    0.05, 0.8, -1.6 ])
        
                # With per-joint limits matching URDF:
        self.torque_limits = np.array([
            20.0, 55.0, 55.0,   # FR hip, upper, lower
            20.0, 55.0, 55.0,   # FL
            20.0, 55.0, 55.0,   # RR
            20.0, 55.0, 55.0,   # RL
        ])

        if GUI_MODE:
            p.connect(p.GUI)  # Opens visual window.
        else:
            # Use for training without visualisation (significantly faster).
            p.connect(p.DIRECT)  # Headless mode (no visuals) → faster for training.

    
        p.configureDebugVisualizer(p.COV_ENABLE_GUI, 0)
        p.resetDebugVisualizerCamera(cameraDistance=1.5,
                                cameraYaw=45,
                                cameraPitch=-30,
                                cameraTargetPosition=[0.3,0,0.1])

        # The action space are the 12 joint angles.
        self.action_space = gym.spaces.Box(low=-1,high=1,shape=(12,),dtype=np.float32)
        self.observation_space = gym.spaces.Box(low=-1, high=1, shape=(SIZE_OBSERVATION,),dtype=np.float32)


    def reward(self, vx, vy, wyaw, torque, joint_vel,action):
        progress = min(self.step_counter_session / MAXIMUM_LENGTH, 1.0)
        # --- 1. Forward velocity (clipped target) ---
        r_forward = min(vx, self.target_speed)

        # ── 2. Cost of Transport — the core efficiency metric ─────────────────────
        # Power = torque × velocity (instantaneous energy use)
        power        = np.abs(torque * joint_vel)
        total_power  = np.sum(power)

        # CoT = power / (mass × g × vx)
        # We use a simplified version — penalize power relative to forward speed
        # If moving fast with little power → low CoT → good
        # If moving slow with lots of power → high CoT → bad
        robot_mass   = 12.0   # A1 mass in kg
        g            = 9.81
        safe_vx      = max(vx, 0.05)   # avoid division by zero
        cot          = total_power / (robot_mass * g * safe_vx)
        r_efficiency = -cot             # minimize CoT

        # --- 2. Lateral + yaw penalty ---
        r_stability = - (vy**2 + wyaw**2)

        # --- 3. Work (energy) ---
        # r_work = -np.abs(np.dot(torque, joint_vel))

        # --- 4. Smoothness (torque change) ---
        r_smooth_tau = -np.sum((torque - self.prev_torque)**2)

        # --- 5. Action smoothness ---
        r_smooth_action = -np.sum((action - self.prev_action)**2)

        # --- 6. Action magnitude ---
        r_action_mag = -np.sum(action**2)

        # --- 7. Joint velocity penalty ---
        r_joint_vel = -np.sum(joint_vel**2)

        # --- 8. Orientation penalty ---
        pos, quat = p.getBasePositionAndOrientation(self.robot_id)
        roll, pitch, _ = p.getEulerFromQuaternion(quat)
        r_orientation = -(roll**2 + pitch**2)

        # --- 9. Vertical velocity penalty ---
        base_vel = p.getBaseVelocity(self.robot_id)[0]
        vz = base_vel[2]
        r_z_vel = -(vz**2)

        # --- 10. Foot contact / slip (simplified) ---
        # If foot in contact → penalize velocity
        foot_vel_penalty = 0
        paw_links = [5, 10, 15, 20]


        for link in paw_links:
            contact = p.getContactPoints(self.robot_id, self.plane_id, link)

            if len(contact) > 0:  # foot in contact
                link_state = p.getLinkState(self.robot_id, link, computeLinkVelocity=1)

                foot_vel = np.array(link_state[6])  # [vx, vy, vz]

                # 🔥 ONLY tangential velocity
                foot_vel_t = foot_vel[:2]   # x, y

                # 🔥 L2 norm (NOT squared)
                foot_vel_penalty += np.linalg.norm(foot_vel_t)

        r_foot_slip = -foot_vel_penalty

            # ── 6. Leg balance — equal work across legs ───────────────────────────────
        work_legs = np.array([
            np.sum(np.abs(torque[0:3]  * joint_vel[0:3])),
            np.sum(np.abs(torque[3:6]  * joint_vel[3:6])),
            np.sum(np.abs(torque[6:9]  * joint_vel[6:9])),
            np.sum(np.abs(torque[9:12] * joint_vel[9:12])),
        ])
        self.work_avg    = 0.99 * self.work_avg + 0.01 * work_legs
        mean_work        = np.mean(self.work_avg)
        r_leg_balance    = -np.sum(np.abs(self.work_avg - mean_work))




        # --- FINAL WEIGHTED SUM ---
        reward = (
            20.0 * r_forward +
            progress*(2.0  * r_stability +
            1.0 * r_efficiency +
            0.001 * r_smooth_tau +
            0.001 * r_smooth_action +
            0.07  * r_action_mag +
            0.002 * r_joint_vel +
            1.5   * r_orientation +
            2.0   * r_z_vel +
            0.8   * r_foot_slip +
            1.0   * r_leg_balance)
        )

        return reward
        
   
    def step(self, action):
        self.step_counter +=1
        if GUI_MODE:
            p.configureDebugVisualizer(p.COV_ENABLE_SINGLE_STEP_RENDERING) ## Render only after 1 step
        # Current joint states
        joint_states = p.getJointStates(self.robot_id, self.joint_id)
        joint_angles = np.array([s[0] for s in joint_states])
        joint_vel = np.array([s[1] for s in joint_states])

        delta_limits = np.array([
            0.10, 0.25, 0.20,   # FR — all reduced to stay within torque limits
            0.10, 0.25, 0.20,   # FL
            0.10, 0.25, 0.20,   # RR
            0.10, 0.25, 0.20,   # RL
        ])
        action = action*delta_limits
        target_angles = self.default_pose + action

        Kp = 200
        Kd = 1.0
        torque = self.motor_strength*(Kp * (target_angles - joint_angles) - Kd * joint_vel)

        torque = np.clip(torque, -self.torque_limits, self.torque_limits)
        p.setJointMotorControlArray(
            self.robot_id,
            self.joint_id,
            controlMode=p.TORQUE_CONTROL,
            forces=torque.tolist()
        )

        for _ in range(2):
            p.stepSimulation()

        base_vel = p.getBaseVelocity(self.robot_id)[0]
        vx = base_vel[0]
        vy = base_vel[1]
    
        _, ang_vel = p.getBaseVelocity(self.robot_id)
        wx, wy, wyaw = ang_vel  # roll rate, pitch rate, yaw rate
        self.vx_smooth = self.alpha_smooth * vx + (1 - self.alpha_smooth) * self.vx_smooth
        self.vy_smooth = self.alpha_smooth * vy + (1 - self.alpha_smooth) * self.vy_smooth
        self.wyaw_smooth = self.alpha_smooth * wyaw + (1 - self.alpha_smooth) * self.wyaw_smooth
        reward = self.reward(self.vx_smooth, self.vy_smooth, self.wyaw_smooth, torque, joint_vel,action)

        # Set state of the current state.
        terminated = False
        truncated = False

        if self.step_counter > self.episode_length:
            self.step_counter_session += self.step_counter
            terminated = False
            truncated = True

        elif self.is_fallen(): # Robot fell
            self.step_counter_session += self.step_counter
            terminated = True
            truncated = False

        observation = self._get_observation()
        self.prev_action = action.copy()
        self.prev_torque = torque.copy()

        # return observation.astype(np.float32), reward, terminated, truncated, {}
        return observation.astype(np.float32), reward, truncated, truncated, {}
    
    def generate_fractal_terrain(self, size=256,
                            scale=1.0,
                            octaves=2,
                            lacunarity=2.0,
                            gain=0.25,
                            frequency=10,
                            amplitude=0.23):

            heightfield = np.zeros((size, size))

            for i in range(size):
                for j in range(size):

                    x = i / size * frequency
                    y = j / size * frequency

                    height = pnoise2(
                        x,
                        y,
                        octaves=octaves,
                        persistence=gain,
                        lacunarity=lacunarity,
                        repeatx=1024,
                        repeaty=1024,
                        base=0
                    )

                    heightfield[i][j] = height

            heightfield = heightfield * amplitude

            return heightfield
    

    def create_heightfield(self, heightfield):

        size = heightfield.shape[0]

        terrainShape = p.createCollisionShape(
            shapeType=p.GEOM_HEIGHTFIELD,
            meshScale=[0.05, 0.05, 1],
            heightfieldTextureScaling=(size - 1) / 2,
            heightfieldData=heightfield.flatten(),
            numHeightfieldRows=size,
            numHeightfieldColumns=size
        )

        terrain = p.createMultiBody(0, terrainShape)

        p.resetBasePositionAndOrientation(terrain, [0, 0, 0], [0,0,0,1])

        return terrain

    def reset(self, seed = None, options=None):
        super().reset(seed=seed)
        # 3. In reset() — add prev_torque reset and increase settle to 300 steps
        self.prev_torque = np.zeros(12)
        # In reset():
        self.contact_avg = np.array([0.5, 0.5, 0.5, 0.5])
        self.work_avg = np.zeros(4)

        self.prev_action = np.zeros(12)
        self.vx_smooth = 0
        self.vy_smooth = 0
        self.wyaw_smooth = 0
        self.step_counter = 0
        p.resetSimulation()
        # Disable rendering during loading.
        p.configureDebugVisualizer(p.COV_ENABLE_RENDERING,0) 
        p.setGravity(0,0,-9.81)
        p.setPhysicsEngineParameter(numSolverIterations=50, numSubSteps=1)
        p.setAdditionalSearchPath(pybullet_data.getDataPath())
        p.setTimeStep(1.0 / 240.0)
        #plane
        # heightfield = self.generate_fractal_terrain(size=256, scale=1, octaves=2, lacunarity=2.0, gain=0.25, frequency=10, amplitude=0.23)
        # plane_id = self.create_heightfield(heightfield)
        self.plane_id = p.loadURDF("plane.urdf")
        # p.changeDynamics(self.plane_id, -1, lateralFriction=5)

        start_pos = [0,0,0.4]
        start_orient = p.getQuaternionFromEuler([0,0,0])

        urdf_path = URDF_PATH
        self.robot_id = p.loadURDF(urdf_path,
                                   start_pos, start_orient,
                                   flags=p.URDF_USE_SELF_COLLISION) ## This will enable the collision checking between robot links
        

        # if np.random.rand() < 0.02:
        #     friction = np.random.uniform(0.6, 1.2)
        #     payload = np.random.uniform(0.0, 0.5)
        #     mass = p.getDynamicsInfo(self.robot_id, -1)[0]

        #     p.changeDynamics(self.robot_id,-1,mass=mass + payload)
        #     p.changeDynamics(plane_id, -1, lateralFriction=friction)
        #     self.motor_strength = np.random.uniform(0.95, 1.05)
        # else:
        self.motor_strength = 1

        # Initialize urdf links and joints.
        self.joint_id = []
        for j in range(p.getNumJoints(self.robot_id)):
            info = p.getJointInfo(self.robot_id, j)
            joint_name = info[1]
            joint_type = info[2]

            if (joint_type == p.JOINT_PRISMATIC
                or joint_type == p.JOINT_REVOLUTE):
                self.joint_id.append(j)


        for j in self.joint_id:
            p.changeDynamics(
                self.robot_id,
                j,
                linearDamping=0.04,

                angularDamping=0.04)
            
         # ADD this before resetJointState
        p.resetBasePositionAndOrientation(
            self.robot_id, [0, 0, 0.40],
            p.getQuaternionFromEuler([0, 0, 0])
        )
        p.resetBaseVelocity(self.robot_id,
            linearVelocity=[0, 0, 0],
            angularVelocity=[0, 0, 0]
        )   
                    

        for j, q in zip(self.joint_id, self.default_pose):
            p.resetJointState(self.robot_id, j, targetValue=q, targetVelocity=0.0)


        # Settle — 100 steps confirmed sufficient at 400 Hz
        for _ in range(100):
            p.setJointMotorControlArray(
                self.robot_id, self.joint_id,
                controlMode=p.POSITION_CONTROL,
                targetPositions=self.default_pose,
                forces        =[200.0] * len(self.joint_id),
                positionGains =[0.5]   * len(self.joint_id),
                velocityGains =[1.0]   * len(self.joint_id),
            )
            p.stepSimulation()

        # Disable motors — hand off to torque control
        p.setJointMotorControlArray(
            self.robot_id, self.joint_id,
            controlMode=p.VELOCITY_CONTROL,
            forces=[0.0] * len(self.joint_id)
        )


        observation = self._get_observation()    

        p.configureDebugVisualizer(p.COV_ENABLE_RENDERING,1)
        return observation.astype(np.float32), {}


    def render(self, mode='human'):
        if GUI_MODE:
            p.configureDebugVisualizer(p.COV_ENABLE_SINGLE_STEP_RENDERING) ## Render only after 1 step



    def close(self):
        p.disconnect()


    def is_fallen(self):
        pos, orient = p.getBasePositionAndOrientation(self.robot_id)
        orient = p.getEulerFromQuaternion(orient)
        is_fallen = (np.fabs(orient[0]) > 0.4
                    or np.fabs(orient[1]) > 0.25
                    or pos[2]<0.1)

        return is_fallen
    
    def _get_observation(self):
        joint_states = p.getJointStates(self.robot_id, self.joint_id)
        # Joint angles
        joint_angles = np.array([s[0] for s in joint_states])
        joint_low = np.array(BOUND_ANGLE_MIN)
        joint_high = np.array(BOUND_ANGLE_MAX)

        joint_angles_norm = 2 * (joint_angles - joint_low) / (joint_high - joint_low) - 1
        joint_angles_norm = np.clip(joint_angles_norm, -1.0, 1.0)
        # Joint velocities
        joint_vel = np.array([s[1] for s in joint_states])
        joint_vel_norm = np.clip(joint_vel / self.max_vel, -1, 1)
        # Base orientation
        pos, quat = p.getBasePositionAndOrientation(self.robot_id)
        roll, pitch, _ = p.getEulerFromQuaternion(quat)
        torso = np.array([roll, pitch])
        torso_norm = torso / np.pi
        # Foot contacts
        paw_links = [5,10,15,20]
        contacts = []
        for idx in paw_links:
            contact = p.getContactPoints(bodyA=self.robot_id,bodyB= self.plane_id, linkIndexA=idx)
            contacts.append(1 if len(contact) > 0 else 0)


        contacts = np.array(contacts)
        observation = np.concatenate(
            (joint_angles_norm,joint_vel_norm,torso_norm,contacts, self.prev_action))

        return observation


In [4]:
GUI_MODE = True

run_model_name      = "repeat2_20000000_steps"
run_full_model_path = os.path.join(TRAINING_MODEL, f"{run_model_name}.zip")
run_full_stats_path = os.path.join(TRAINING_MODEL, f"repeat2_vecnormalize_20000000_steps.pkl")

# run_model_name = "again8"
# run_full_model_path = os.path.join(SAVE_PATH, f"{run_model_name}.zip")
# run_full_stats_path = os.path.join(SAVE_PATH, f"stats_{run_model_name}.pkl")

eval_env = make_vec_env(OpenCatGymEnv, n_envs=1)
eval_env = VecNormalize.load(run_full_stats_path, eval_env)
eval_env.training    = False
eval_env.norm_reward = False

model    = PPO.load(run_full_model_path, env=eval_env)
obs      = eval_env.reset()
real_env = eval_env.envs[0].unwrapped

vx_log        = []
contact_log   = []
roll_log      = []
joint_vel_log = []
energy_log = []
distance_log = []

total_energy = 0
total_distance = 0

dt = 1.0 / 240.0
mass = 12.0  # approx Unitree A1 mass (adjust if needed)

for i in range(1000):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, info = eval_env.step(action)

    lv, _          = p.getBaseVelocity(real_env.robot_id)
    _, orn         = p.getBasePositionAndOrientation(real_env.robot_id)
    roll, pitch, _ = p.getEulerFromQuaternion(orn)
    contacts       = [
        1 if len(p.getContactPoints(
            bodyA=real_env.robot_id,
            bodyB=real_env.plane_id,
            linkIndexA=c)) > 0 else 0
        for c in [5, 10, 15, 20]
    ]
    states   = p.getJointStates(real_env.robot_id, real_env.joint_id)
    jvel     = np.array([s[1] for s in states])
    # Compute torque again (same as env)
    torque = real_env.prev_torque  

    instant_power = np.sum(np.abs(torque * jvel))
    total_energy += instant_power * dt
    total_distance += lv[0] * dt


    vx_log.append(lv[0])
    contact_log.append(contacts)
    roll_log.append(np.degrees(roll))
    joint_vel_log.append(jvel)

    if (i + 1) % 1000 == 0:
        obs      = eval_env.reset()
        real_env = eval_env.envs[0].unwrapped
        print(i)

eval_env.close()

contact_log   = np.array(contact_log)   # shape (2000, 4)
joint_vel_log = np.array(joint_vel_log)

energy = total_energy
cot = energy / (mass * 9.81 * total_distance + 1e-6)

# ── Basic metrics ─────────────────────────────────────────────────────────────
print("── Gait Quality Report ──────────────────────────────────")
print(f"  Mean vx    : {np.mean(vx_log):.3f} m/s")
print(f"  % forward  : {np.mean(np.array(vx_log)>0.1)*100:.1f}%")
print(f"  Mean roll  : {np.mean(np.abs(roll_log)):.2f} deg")
print(f"  Max roll   : {np.max(np.abs(roll_log)):.2f} deg")

print(f"\n── Energy Metrics ─────────────────────────────")
print(f"  Total Energy     : {energy:.2f} J")
print(f"  Distance         : {total_distance:.2f} m")
print(f"  Cost of Transport: {cot:.3f}")

# ── Foot contact rates ────────────────────────────────────────────────────────
print(f"\n── Foot contact rates ───────────────────────────────────")
names = ["FR", "FL", "RR", "RL"]
rates = {}
for i, name in enumerate(names):
    rates[name] = np.mean(contact_log[:, i]) * 100
    print(f"  {name}: {rates[name]:.1f}%")

# ── Pair synchrony — key to gait identification ───────────────────────────────
# FR=0, FL=1, RR=2, RL=3
fr_rl_sync = np.mean(contact_log[:,0] == contact_log[:,3]) * 100  # diagonal
fl_rr_sync = np.mean(contact_log[:,1] == contact_log[:,2]) * 100  # diagonal
fr_fl_sync = np.mean(contact_log[:,0] == contact_log[:,1]) * 100  # front pair
rr_rl_sync = np.mean(contact_log[:,2] == contact_log[:,3]) * 100  # rear pair
fr_rr_sync = np.mean(contact_log[:,0] == contact_log[:,2]) * 100  # right pair
fl_rl_sync = np.mean(contact_log[:,1] == contact_log[:,3]) * 100  # left pair

# How often ALL feet are on ground simultaneously
all_contact  = np.mean(np.all(contact_log == 1, axis=1)) * 100
# How often NO feet are on ground (aerial phase)
no_contact   = np.mean(np.all(contact_log == 0, axis=1)) * 100
# Mean number of feet on ground at any time
mean_feet_on = np.mean(np.sum(contact_log, axis=1))

print(f"\n── Synchrony analysis ───────────────────────────────────")
print(f"  FR+RL diagonal sync : {fr_rl_sync:.1f}%")
print(f"  FL+RR diagonal sync : {fl_rr_sync:.1f}%")
print(f"  FR+FL front sync    : {fr_fl_sync:.1f}%")
print(f"  RR+RL rear sync     : {rr_rl_sync:.1f}%")
print(f"  FR+RR right sync    : {fr_rr_sync:.1f}%")
print(f"  FL+RL left sync     : {fl_rl_sync:.1f}%")
print(f"\n  All 4 feet on ground: {all_contact:.1f}%")
print(f"  No feet on ground   : {no_contact:.1f}%  (aerial phase)")
print(f"  Mean feet on ground : {mean_feet_on:.2f} / 4")

# ── Gait identification ───────────────────────────────────────────────────────
print(f"\n── Gait Identification ──────────────────────────────────")

# Gait patterns and their signatures:
#
# TROT:    diagonal pairs move together
#          FR+RL sync ≈ 50%, FL+RR sync ≈ 50%
#          mean_feet_on ≈ 2.0, no aerial phase
#
# WALK:    each foot moves independently, one at a time
#          no pair is strongly synced
#          mean_feet_on ≈ 3.0 (3 feet on ground always)
#          all_contact high, no aerial phase
#
# BOUND:   front pair together, rear pair together
#          FR+FL sync high, RR+RL sync high
#          aerial phase present
#
# PACE:    same-side pairs together (left legs, right legs)
#          FR+RR sync high, FL+RL sync high
#
# GALLOP:  one foot at a time, sequential
#          low synchrony on all pairs
#          aerial phase present, mean_feet_on < 2
#
# PRONK:   all 4 feet together
#          all_contact very high
#
# SHUFFLE: mostly all on ground, little lift
#          all_contact > 60%, mean_feet_on > 3.5

diagonal_score = (fr_rl_sync + fl_rr_sync) / 2
front_rear_score = (fr_fl_sync + rr_rl_sync) / 2
lateral_score  = (fr_rr_sync + fl_rl_sync) / 2

scores = {
    "TROT"    : diagonal_score,
    "BOUND"   : front_rear_score,
    "PACE"    : lateral_score,
    "WALK"    : 100 - min(diagonal_score, front_rear_score, lateral_score),
    "PRONK"   : all_contact,
    "SHUFFLE" : all_contact if mean_feet_on > 3.0 else 0,
    "GALLOP"  : no_contact * 2 if mean_feet_on < 2.0 else 0,
}

# Apply additional heuristics
if mean_feet_on > 3.2:
    detected = "SHUFFLE (robot barely lifting feet)"
elif all_contact > 50:
    detected = "PRONK (all feet together)"
elif no_contact > 10 and front_rear_score > 60:
    detected = "BOUND (front+rear pairs, aerial phase)"
elif no_contact > 10 and mean_feet_on < 2.0:
    detected = "GALLOP (sequential, aerial phase)"
elif lateral_score > 65:
    detected = "PACE (same-side legs together)"
elif diagonal_score > 55 and mean_feet_on < 2.5:
    detected = "TROT (diagonal pairs)"
elif mean_feet_on > 2.5 and all_contact < 30:
    detected = "WALK (3 feet on ground, sequential)"
else:
    detected = "MIXED / ASYMMETRIC"

print(f"\n  Diagonal score   : {diagonal_score:.1f}   (TROT signature)")
print(f"  Front/rear score : {front_rear_score:.1f}   (BOUND signature)")
print(f"  Lateral score    : {lateral_score:.1f}   (PACE signature)")
print(f"  Mean feet on     : {mean_feet_on:.2f}  (WALK=3+, TROT=2, GALLOP<2)")
print(f"  Aerial phase     : {no_contact:.1f}%  (BOUND/GALLOP have this)")
print(f"\n  >>> Detected gait: {detected}")

# ── Joint activity ────────────────────────────────────────────────────────────
print(f"\n── Joint activity (mean |velocity|) ─────────────────────")
mean_jvel = np.mean(np.abs(joint_vel_log), axis=0)
for i, name in enumerate(["FR_hip","FR_thigh","FR_knee",
                           "FL_hip","FL_thigh","FL_knee",
                           "RR_hip","RR_thigh","RR_knee",
                           "RL_hip","RL_thigh","RL_knee"]):
    bar = "█" * int(mean_jvel[i] * 5)
    print(f"  {name:12s}: {mean_jvel[i]:.3f}  {bar}")

front_activity = np.mean(mean_jvel[:6])
rear_activity  = np.mean(mean_jvel[6:])
ratio          = front_activity / rear_activity if rear_activity > 0 else float("inf")
print(f"\n  Front leg activity : {front_activity:.3f}")
print(f"  Rear  leg activity : {rear_activity:.3f}")
print(f"  Symmetry ratio     : {ratio:.2f}x  (1.0 = perfect)")
if ratio > 2.0:
    print("  ⚠ Front legs dominant — rear legs dragging")
elif ratio < 0.5:
    print("  ⚠ Rear legs dominant — front legs passive")
else:
    print("  ✓ Relatively symmetric")
print("─────────────────────────────────────────────────────────")
# ```

# The gait identification logic checks six signatures:
# ```
# TROT    → diagonal pairs (FR+RL, FL+RR) in sync ~50%
# WALK    → 3 feet on ground at all times, sequential
# BOUND   → front pair + rear pair in sync, aerial phase
# PACE    → same-side pairs (FR+RR, FL+RL) in sync
# GALLOP  → sequential footfalls, aerial phase, <2 feet on ground
# PRONK   → all 4 feet together
# SHUFFLE → >3.2 feet on ground at all times, barely lifting

999
── Gait Quality Report ──────────────────────────────────
  Mean vx    : 0.373 m/s
  % forward  : 100.0%
  Mean roll  : 0.80 deg
  Max roll   : 3.38 deg

── Energy Metrics ─────────────────────────────
  Total Energy     : 519.93 J
  Distance         : 1.55 m
  Cost of Transport: 2.841

── Foot contact rates ───────────────────────────────────
  FR: 46.2%
  FL: 47.6%
  RR: 39.4%
  RL: 36.9%

── Synchrony analysis ───────────────────────────────────
  FR+RL diagonal sync : 60.3%
  FL+RR diagonal sync : 84.8%
  FR+FL front sync    : 89.8%
  RR+RL rear sync     : 69.9%
  FR+RR right sync    : 85.6%
  FL+RL left sync     : 65.1%

  All 4 feet on ground: 19.5%
  No feet on ground   : 35.9%  (aerial phase)
  Mean feet on ground : 1.70 / 4

── Gait Identification ──────────────────────────────────

  Diagonal score   : 72.5   (TROT signature)
  Front/rear score : 79.8   (BOUND signature)
  Lateral score    : 75.3   (PACE signature)
  Mean feet on     : 1.70  (WALK=3+, TROT=2, GALLOP<2)
  